# 02 · Khine: CatBoost, LightGBM and an ensemble

Notebook 01 showed that hand-made features and tuning do not move `HistGradientBoosting` past the baseline
(validation AUC 0.95758). This notebook tries **different, stronger models** instead, then averages them.

1. Baseline gradient boosting again (for the ensemble)
2. CatBoost
3. LightGBM
4. Ensemble: weighted average of ranks, weights picked on the validation split
5. Refit the chosen models on all rows, save a v3 submission

Same team split (`make_split`) as everyone else, so the scores compare. The baseline alone is always one of the
ensemble options, so this notebook never picks something worse than v1 on validation.

Note: CatBoost and LightGBM use the validation split for early stopping, which makes their validation AUC very
slightly optimistic. The Kaggle score is the real check.

Runtime: roughly 20–40 minutes on a laptop (CatBoost is the slow part).

In [1]:
import sys, time, itertools
sys.path.append('..')          # so we can import src/ from the notebooks folder

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier, Pool
import lightgbm as lgb

from src.clean import load_data, make_features, make_split, save_submission, CATEGORICAL, RATINGS, TARGET, ID

## 1. Data, split and leaderboard

In [2]:
train, test = load_data()
lb = pd.read_csv('../report/leaderboard_private_2026-11-26.csv')
lb = lb[lb['Score'] > 0.5]
def rank(auc):
    return int((lb['Score'] > auc).sum()) + 1

# Baseline features (one-hot text columns) and the team split
X, y, X_test = make_features(train, test, ratings_as_categories=False)
X_tr, X_va, y_tr, y_va = make_split(X, y)
tr, va = X_tr.index, X_va.index

# CatBoost / LightGBM handle text columns themselves, so give them the raw columns
# (same missing-value fix as src/clean.py; trees don't need the log transform)
def raw_features(df):
    df = df.drop(columns=[c for c in (ID, TARGET) if c in df.columns]).copy()
    df['Arrival Delay in Minutes'] = df['Arrival Delay in Minutes'].fillna(df['Departure Delay in Minutes'])
    return df

R, R_test = raw_features(train), raw_features(test)
print('train', R.shape, ' test', R_test.shape, ' validation rows', len(va))

val_pred, results, n_trees = {}, {}, {}
def log(name, p, trees):
    val_pred[name], n_trees[name] = p, trees
    results[name] = roc_auc_score(y_va, p)
    print(f'{name}: AUC {results[name]:.5f} · {trees} trees')

train (699635, 21)  test (299844, 21)  validation rows 139927


## 2. Baseline gradient boosting (same settings as `00_baseline`)

In [3]:
t = time.time()
hgb = HistGradientBoostingClassifier(max_iter=2000, learning_rate=0.05, max_leaf_nodes=63,
                                     early_stopping=True, n_iter_no_change=50, random_state=0).fit(X_tr, y_tr)
log('hgb', hgb.predict_proba(X_va)[:, 1], hgb.n_iter_)
print(f'{time.time()-t:.0f}s')

hgb: AUC 0.95758 · 398 trees
114s


## 3. CatBoost
CatBoost turns the text columns (Gender, Customer Type, Type of Travel, Class) into numbers in a smart way
(target statistics). The 0–5 ratings stay numbers.

In [4]:
def fit_catboost(cat_cols, **params):
    settings = dict(iterations=4000, learning_rate=0.08, depth=6, eval_metric='AUC', metric_period=25,
                    od_type='Iter', od_wait=200, random_seed=0, thread_count=-1, verbose=500,
                    allow_writing_files=False)
    settings.update(params)
    t = time.time()
    model = CatBoostClassifier(**settings)
    model.fit(Pool(R.loc[tr], y_tr, cat_features=cat_cols),
              eval_set=Pool(R.loc[va], y_va, cat_features=cat_cols), use_best_model=True)
    print(f'{time.time()-t:.0f}s')
    return model

cb_num = fit_catboost(CATEGORICAL)
log('catboost_ratings_numbers', cb_num.predict_proba(R.loc[va])[:, 1], cb_num.get_best_iteration() + 1)

0:	test: 0.9337921	best: 0.9337921 (0)	total: 2.84s	remaining: 3h 9m 25s


500:	test: 0.9571005	best: 0.9571005 (500)	total: 6m 22s	remaining: 44m 28s


1000:	test: 0.9573899	best: 0.9573925 (999)	total: 12m 12s	remaining: 36m 35s


1500:	test: 0.9574895	best: 0.9574958 (1492)	total: 18m 53s	remaining: 31m 26s


Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.9575477699
bestIteration = 1758

Shrink model to first 1759 iterations.


1589s


catboost_ratings_numbers: AUC 0.95755 · 1759 trees


We also tried the 13 ratings as CatBoost **categories**. On a 20,000-row test it scored lower
(0.9523 vs 0.9536 with ratings as numbers) and was about 3× slower. A first full-data run that included
it went over an hour without finishing, so it is left out.

In [5]:
cb_name, cb_cols, cb_extra = 'catboost_ratings_numbers', CATEGORICAL, {}

## 4. LightGBM

In [6]:
def to_lgb(df):
    df = df.copy()
    for c in CATEGORICAL:
        df[c] = pd.Categorical(df[c], categories=sorted(R[c].unique()))
    return df

L, L_test = to_lgb(R), to_lgb(R_test)
lgb_params = dict(learning_rate=0.03, num_leaves=63, min_child_samples=50, subsample=0.8, subsample_freq=1,
                  colsample_bytree=0.7, reg_lambda=1.0, random_state=0, verbose=-1)
t = time.time()
lgbm = lgb.LGBMClassifier(n_estimators=5000, **lgb_params)
lgbm.fit(L.loc[tr], y_tr, eval_set=[(L.loc[va], y_va)], eval_metric='auc',
         callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(500)])
log('lightgbm', lgbm.predict_proba(L.loc[va])[:, 1], lgbm.best_iteration_)
print(f'{time.time()-t:.0f}s')

C:\Users\khine\Downloads\cs4650-capstone-airline-starter\cs4650-capstone-airline\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[500]	valid_0's auc: 0.95808	valid_0's binary_logloss: 0.229639


[1000]	valid_0's auc: 0.95816	valid_0's binary_logloss: 0.229212


lightgbm: AUC 0.95819 · 842 trees
217s


## 5. Ensemble
AUC only cares about the order of predictions, so we average **ranks**. We try every weight combination in
steps of 0.1 (weights add up to 1). The baseline alone (weight 1, 0, 0) is one of the options.

In [7]:
members = ['hgb', cb_name, 'lightgbm']
ranks = {m: pd.Series(val_pred[m]).rank(pct=True).to_numpy() for m in members}

steps = [round(0.1 * i, 1) for i in range(11)]
grid = [w for w in itertools.product(steps, repeat=3) if abs(sum(w) - 1) < 1e-9]
scores = {w: roc_auc_score(y_va, sum(wi * ranks[m] for wi, m in zip(w, members))) for w in grid}
best_w = max(scores, key=scores.get)
equal = roc_auc_score(y_va, sum(ranks[m] for m in members) / 3)

results['ensemble, equal weights'] = equal
results[f'ensemble, weights {dict(zip(members, best_w))}'] = scores[best_w]
print('best weights:', dict(zip(members, best_w)), f'AUC {scores[best_w]:.5f}')
print(f'equal weights: AUC {equal:.5f}')

best weights: {'hgb': 0.1, 'catboost_ratings_numbers': 0.2, 'lightgbm': 0.7} AUC 0.95824
equal weights: AUC 0.95816


## 6. Results
"Would rank" compares validation AUC with the final private leaderboard. For v1 the private score
(0.95908) came out about 0.0015 higher than validation, so these ranks are on the cautious side.

In [8]:
table = pd.DataFrame({'validation AUC': results}).round(5)
table['vs baseline'] = (table['validation AUC'] - results['hgb']).map('{:+.5f}'.format)
table['would rank'] = [f'{rank(a):,} / {len(lb):,}' for a in table['validation AUC']]
table

,validation AUC,vs baseline,would rank
hgb,0.95758,-0.00000,"3,364 / 3,715"
catboost_ratings_numbers,0.95755,-0.00003,"3,367 / 3,715"
lightgbm,0.95819,+0.00061,"3,279 / 3,715"
"ensemble, equal weights",0.95816,+0.00058,"3,282 / 3,715"
"ensemble, weights {'hgb': 0.1, 'catboost_ratings_numbers': 0.2, 'lightgbm': 0.7}",0.95824,+0.00066,"3,264 / 3,715"


## 7. Submission file
Refit every model with weight > 0 on **all** training rows, using the number of trees found above.
If the best option is the baseline alone, there is nothing new to submit.

In [9]:
weights = {m: w for m, w in zip(members, best_w) if w > 0}
gain = scores[best_w] - results['hgb']
print('weights:', weights, f' gain over baseline: {gain:+.5f}')

if set(weights) == {'hgb'}:
    print('The baseline alone is best: no new submission.')
else:
    test_pred = {}
    if 'hgb' in weights:
        m = HistGradientBoostingClassifier(max_iter=n_trees['hgb'], learning_rate=0.05, max_leaf_nodes=63,
                                           early_stopping=False, random_state=0).fit(X, y)
        test_pred['hgb'] = m.predict_proba(X_test)[:, 1]
    if cb_name in weights:
        t = time.time()
        m = CatBoostClassifier(iterations=n_trees[cb_name], learning_rate=0.08, depth=6,
                               random_seed=0, thread_count=-1, verbose=0, allow_writing_files=False, **cb_extra)
        m.fit(Pool(R, y, cat_features=cb_cols))
        test_pred[cb_name] = m.predict_proba(R_test)[:, 1]
        print(f'catboost refit {time.time()-t:.0f}s')
    if 'lightgbm' in weights:
        m = lgb.LGBMClassifier(n_estimators=n_trees['lightgbm'], **lgb_params).fit(L, y)
        test_pred['lightgbm'] = m.predict_proba(L_test)[:, 1]

    p_test = sum(w * pd.Series(test_pred[m]).rank(pct=True).to_numpy() for m, w in weights.items())
    print(save_submission(test, p_test, 'v3_ensemble'))

    short = {'hgb': 'HistGB', 'catboost_ratings_numbers': 'CatBoost (ratings numeric)',
             'catboost_ratings_categories': 'CatBoost (ratings categorical)', 'lightgbm': 'LightGBM'}
    parts = ' + '.join(f'{w:.1f} {short[m]} ({n_trees[m]} trees)' for m, w in weights.items())
    print('\nKaggle description:')
    print(f'v3 - Rank-average ensemble: {parts}. New models vs v1 baseline. Validation AUC {scores[best_w]:.5f}.')

weights:

 {'hgb': 0.1, 'catboost_ratings_numbers': 0.2, 'lightgbm': 0.7}  gain over baseline: +0.00065


catboost refit 833s


saved submissions\v3_ensemble.csv

Kaggle description:
v3 - Rank-average ensemble: 0.1 HistGB (398 trees) + 0.2 CatBoost (ratings numeric) (1759 trees) + 0.7 LightGBM (842 trees). New models vs v1 baseline. Validation AUC 0.95824.


## After running
1. Upload `submissions/v3_ensemble.csv` with **Late Submission**, using the description printed above.
2. Add a row to the README results log.